In [0]:
# ============================================================
# Pipeline : covid19_medallion_pipeline
# Notebook : 05_facts/06_fact_payer_coverage_period
# Purpose  : fact_payer_coverage_period - Grain: One row = one insurance
#            coverage period for a specific patient (renamed from the legacy
#            "transitions" since source lacks from_payer/to_payer fields,
#            representing coverage periods instead).
# ============================================================

import dlt
from pyspark.sql import functions as F


@dlt.table(
    name="covid19_lakehouse.gold.fact_payer_coverage_period",
    comment="Fact table - one row per patient-payer coverage period",
)
def fact_payer_coverage_period():
    coverage = dlt.read_stream("covid19_lakehouse.silver.payer_coverage_period")

    patients = (
        spark.read.table("covid19_lakehouse.gold.dim_patient")
        .filter(F.col("is_current") == True)  # noqa: E712
        .select(F.col("patient_id"), F.col("patient_key"))
    )
    payers = spark.read.table("covid19_lakehouse.gold.dim_payer").select(
        F.col("payer_id"), F.col("payer_key")
    )

    df = coverage.join(patients, on="patient_id", how="left").join(
        payers, on="payer_id", how="left"
    )

    return df.select(
        F.xxhash64("patient_id", "payer_id", "start_year").alias("coverage_fact_key"),
        F.col("patient_key"),
        F.col("payer_key"),
        F.col("start_year"),
        F.col("end_year"),
        F.col("ownership"),
        F.col("duration_years"),
    )